# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` as per Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL and complies with the FAIR principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is a CroissantMetadata object

# Display basic information
print(f"Dataset Name       : {metadata.name}")
print(f"Version            : {getattr(metadata, 'version', 'N/A')}")
print(f"Published          : {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Description        : {metadata.description}")
print(f"Citation           : {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, their IDs, and available fields/columns by their `@id`.

First, we enumerate all record sets in the package, then for each, list its available fields.

In [ ]:
# List all record sets (`RecordSet`) and their fields (`Field`) by `@id`

record_set_objs = getattr(metadata, 'recordSet', [])

if not record_set_objs:
    # Try `record_sets` field (old mlcroissant internal) fallback
    record_set_objs = getattr(metadata, 'record_sets', [])

# Gather `@id`s for all record sets
record_set_ids = []

print("Available Record Sets and their Fields:\n")
for rs in record_set_objs:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', 'UNKNOWN'))
    rs_name = getattr(rs, 'name', 'N/A')
    print(f"- Record Set: {rs_name} (@id={rs_id})")
    record_set_ids.append(rs_id)
    field_objs = getattr(rs, 'field', getattr(rs, 'fields', []))
    if not field_objs:
        print("  No field objects available.")
        continue
    for fld in field_objs:
        f_id = getattr(fld, '@id', getattr(fld, 'id', 'UNKNOWN'))
        f_name = getattr(fld, 'name', 'N/A')
        print(f"    - Field: {f_name} (@id={f_id})")
print("\nIf no record sets are shown above, please check the latest dataset schema.")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.

We reference record sets and fields by their `@id`. This code will load each record set (typically tables or record groups) into a dataframe. Adjust the record set `@id`s as appropriate for your dataset.

In [ ]:
# Example: Load data from all detected record sets.
dataframes = {}

if not record_set_ids:
    print("No record sets detected for extraction. Please consult the Croissant schema and update record_set_ids.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set @id='{record_set_id}' ...")
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Data shape: {dataframes[record_set_id].shape}")
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
    # For demonstration, pick the first record set to show a head preview
    selected_record_set_id = record_set_ids[0]
    print(f"\nPreview of first rows from record set: {selected_record_set_id}\n")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter by a numeric field, normalize, and group by another attribute. All fields are referenced via their `@id`.

> ⚠️ Please set `numeric_field_id` and `group_field_id` below to valid field `@id`s as shown in the overview above for this dataset.

In [ ]:
# Specify the record set and fields for EDA by their `@id`

# Example values -- replace with the correct @id from your schema as seen above
# Example: numeric_field_id = "cr:peptideCount"  (fictional example, adjust to your schema)
# Example: group_field_id = "cr:sampleType"

record_set_id = selected_record_set_id  # Use the first record set loaded above
numeric_field_id = None  # e.g., "cr:coveragePercent", update with a field @id
group_field_id = None   # e.g., "cr:sampleType", update with a field @id

df = dataframes.get(record_set_id)

if df is not None and not df.empty:
    # Try to auto-detect a numeric field if not supplied
    if numeric_field_id is None:
        # Guess first numeric column
        possible_numeric = df.select_dtypes(include='number').columns
        if len(possible_numeric) > 0:
            numeric_field_id = possible_numeric[0]
        else:
            print("No numeric field detected. Please set numeric_field_id explicitly.")
            numeric_field_id = None
    if group_field_id is None:
        # Guess a candidate group field (first non-numeric, if available)
        possible_group = df.select_dtypes(exclude='number').columns
        if len(possible_group) > 0:
            group_field_id = possible_group[0]
        else:
            print("No group field detected. Please set group_field_id explicitly.")
            group_field_id = None
    
    print(f"Using field '@id': {numeric_field_id} for numeric analysis.")
    print(f"Using field '@id': {group_field_id} for grouping (if applicable).")
    
    # Filter by threshold
    threshold = 10
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered to {len(filtered_df)} rows where {numeric_field_id} > {threshold}.")
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped (mean) by {group_field_id}:")
            print(grouped_df[[numeric_field_id]].head())
    else:
        print(f"Numeric field {numeric_field_id} not found in columns: {df.columns.tolist()}")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Replace variable names with those found in your dataset as appropriate. Ensure the field IDs in the plots reflect their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze a Croissant-FAIR dataset using `mlcroissant` and referencing all dataset elements by their unique `@id`. For in-depth analyses, adjust the record set and field IDs above to match the biological questions or target analytes of interest in your project.

For more details on the dataset, refer to its Croissant schema or associated publication.